In [0]:
df_order_payments = spark.table("ecommerce_dev.bronze.order_payments")

df_order_payments.printSchema()
df_order_payments.limit(5).display()
print(f"Row count: {df_order_payments.count()}")

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,_ingested_at,_source_file,_source_table
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments


Row count: 103886


In [0]:
from pyspark.sql.functions import *

# Nulls per column
df_order_payments.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df_order_payments.columns]
).display()

# Full-row duplicates
print("Full duplicate rows:", df_order_payments.count() - df_order_payments.dropDuplicates().count())

# composite key check
print("Duplicate (order_id, payment_sequential):",
      df_order_payments.count() - df_order_payments.dropDuplicates(["order_id", "payment_sequential"]).count())

# check _rescued_data
df_order_payments.select("_rescued_data").filter(col("_rescued_data").isNotNull()).display()

# distinct payment types
df_order_payments.groupBy("payment_type").count().orderBy(desc("count")).display()

# installments and value sanity
df_order_payments.select(
    min("payment_installments"), max("payment_installments"),
    min("payment_value"), max("payment_value")
).display()

print("Rows with payment_value <= 0:", df_order_payments.filter(col("payment_value") <= 0).count())
print("Rows with payment_installments <= 0:", df_order_payments.filter(col("payment_installments") <= 0).count())

# FK check against orders
print("order_payments with order_id not in orders:",
      df_order_payments.join(spark.table("ecommerce_dev.silver.orders"), "order_id", "left_anti").count())

order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,_ingested_at,_source_file,_source_table
0,0,0,0,0,103886,0,0,0


Full duplicate rows: 0
Duplicate (order_id, payment_sequential): 0


_rescued_data


payment_type,count
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


min(payment_installments),max(payment_installments),min(payment_value),max(payment_value)
0,24,0.0,13664.08


Rows with payment_value <= 0: 9
Rows with payment_installments <= 0: 2
order_payments with order_id not in orders: 0


In [0]:
from pyspark.sql.functions import *

# the not_defined payment type rows
df_order_payments.filter(col("payment_type") == "not_defined").display()

# zero/negative payment_value rows
df_order_payments.filter(col("payment_value") <= 0).display()

# zero/negative installments rows
df_order_payments.filter(col("payment_installments") <= 0).display()

order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,_ingested_at,_source_file,_source_table
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments


order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,_ingested_at,_source_file,_source_table
8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments


order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,_ingested_at,_source_file,_source_table
744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments
1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94,null,2026-08-04T00:55:26.069Z,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv,order_payments


In [0]:
from pyspark.sql.functions import *

df_silver_order_payments = (
    df_order_payments
    # flag installments=0 as a data quality issue rather than silently fixing
    .withColumn("has_invalid_installments", col("payment_installments") == 0)
    .drop("_rescued_data", "_source_file", "_source_table")
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
)

df_silver_order_payments.limit(5).display()
print("Flagged invalid installments:", df_silver_order_payments.filter(col("has_invalid_installments")).count())

order_id,payment_sequential,payment_type,payment_installments,payment_value,bronze_ingested_at,has_invalid_installments
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-08-04T00:55:26.069Z,false
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-08-04T00:55:26.069Z,false
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-08-04T00:55:26.069Z,false
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-08-04T00:55:26.069Z,false
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-08-04T00:55:26.069Z,false


Flagged invalid installments: 2


In [0]:
(df_silver_order_payments.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("ecommerce_dev.silver.order_payments"))

In [0]:
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_payments
    ALTER COLUMN order_id SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_payments
    ALTER COLUMN payment_sequential SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_payments
    ADD CONSTRAINT pk_order_payments PRIMARY KEY (order_id, payment_sequential)
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.order_payments
    ADD CONSTRAINT fk_order_payments_order FOREIGN KEY (order_id)
    REFERENCES ecommerce_dev.silver.orders (order_id)
""")

DataFrame[]

In [0]:
spark.sql("""
    COMMENT ON TABLE ecommerce_dev.silver.order_payments IS
    'Cleaned payments table. Composite PK: (order_id, payment_sequential). FK: order_id -> silver.orders. payment_value=0 is legitimate for voucher/not_defined types (kept as-is). payment_installments=0 flagged via has_invalid_installments (2 rows, real anomaly, not fabricated).'
""")

DataFrame[]